# Data Preprocessing
Load, clean, and encode the raw cars dataset.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('cars.csv')
print(df.shape)
print(df.isnull().sum())
df.head(2)

(762091, 20)
manufacturer                0
model                       0
year                        0
mileage                   506
engine                  15050
transmission             9904
drivetrain              21562
fuel_type               22927
mpg                    142071
exterior_color           8859
interior_color          56975
accidents_or_damage     24212
one_owner               31483
personal_use_only       24852
seller_name              8593
seller_rating          213973
driver_rating           31632
driver_reviews_num          0
price_drop             351979
price                       0
dtype: int64


,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0
1,Acura,ILX Hybrid 1.5L,2013,47645.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Gray,Ebony,1.0,1.0,1.0,Kars Today,NaN,4.4,12.0,NaN,17995.0


In [3]:
df.drop(columns=['model', 'seller_name', 'exterior_color', 'interior_color'], inplace=True)

In [4]:
def parse_mpg(val):
    if pd.isna(val):
        return np.nan
    nums = []
    for part in str(val).replace('-', ' ').split():
        try:
            nums.append(float(part))
        except:
            pass
    return np.mean(nums) if nums else np.nan

df['mpg'] = df['mpg'].apply(parse_mpg)

In [5]:
def extract_displacement(text):
    for part in str(text).split():
        if part.upper().endswith('L') and len(part) > 1:
            try:
                val = float(part[:-1])
                if 0.5 <= val <= 9.0:
                    return val
            except:
                pass
    return np.nan

def extract_hp(text):
    upper = str(text).upper().replace('-', ' ')
    parts = upper.split()
    for i, part in enumerate(parts):
        if part == 'HP' and i > 0:
            try:
                return float(parts[i - 1])
            except:
                pass
        elif part.endswith('HP') and len(part) > 2:
            try:
                return float(part[:-2])
            except:
                pass
    return np.nan

df['engine_displacement'] = df['engine'].apply(extract_displacement)
df['engine_hp'] = df['engine'].apply(extract_hp)
df.drop(columns=['engine'], inplace=True)

In [6]:
df['car_age'] = 2024 - df['year']
df.drop(columns=['year'], inplace=True)

In [7]:
p01, p99 = df['price'].quantile([0.01, 0.99])
df = df[(df['price'] >= p01) & (df['price'] <= p99)].reset_index(drop=True)
print(f'After outlier removal: {df.shape}')

After outlier removal: (746906, 17)


In [8]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [9]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

df.head(2)

,manufacturer,mileage,transmission,drivetrain,fuel_type,mpg,accidents_or_damage,one_owner,personal_use_only,seller_rating,driver_rating,driver_reviews_num,price_drop,price,engine_displacement,engine_hp,car_age
0,0,92945.0,845,26,17,38.5,0.0,0.0,0.0,4.5,4.4,12.0,300.0,13988.0,1.5,90.0,11
1,0,47645.0,862,26,20,38.5,1.0,1.0,1.0,4.5,4.4,12.0,627.0,17995.0,1.5,260.0,11


In [10]:
df.to_csv('preprocessed.csv', index=False)
print(f'Saved: {df.shape}')

Saved: (746906, 17)
